In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [4]:
def search_douban_mobile(title):
    search_url = f"https://m.douban.com/search/?query={title}"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(search_url, headers=headers)
    return BeautifulSoup(response.text, "html.parser")

In [5]:
def extract_top_results(soup):
    results = []
    for li in soup.select("ul.search_results_subjects > li")[:5]:
        a_tag = li.find("a", href=True)
        if not a_tag:
            continue
        href = urljoin("https://m.douban.com", a_tag["href"])
        subject_title = li.select_one(".subject-title")
        results.append({
            "title": subject_title.text.strip() if subject_title else "",
            "url": href
        })
    return results

In [6]:
def find_exact_match(results, target_title):
    for r in results:
        if r["title"] == target_title:
            return r["url"]
    return None

In [11]:
def parse_douban_info(url):
    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
    soup = BeautifulSoup(html, "html.parser")
    info_div = soup.find("div", id="info")
    if not info_div:
        return {}

    def extract_text(label):
        tag = info_div.find("span", string=lambda s: s and s.strip() == label)
        if tag and tag.next_sibling:
            return tag.next_sibling.get_text(strip=True)
        return None

    def extract_list(label):
        tag = info_div.find("span", string=lambda s: s and s.strip() == label)
        if tag:
            attrs = tag.find_next("span", class_="attrs")
            if attrs:
                return [a.get_text(strip=True) for a in attrs.find_all("a")]
        return []
    
    # Rating and review count
    rating_block = soup.find("div", class_="rating_self")
    rating = None
    review_count = None
    if rating_block:
        rating_tag = rating_block.find("strong", class_="rating_num")
        votes_tag = rating_block.find("span", property="v:votes")
        rating = rating_tag.text.strip() if rating_tag else None
        review_count = votes_tag.text.strip() if votes_tag else None


    metadata = {
        "title": soup.select_one("span[property='v:itemreviewed']").text.strip() if soup.select_one("span[property='v:itemreviewed']") else None,
        "matched_url": url,
        "director": soup.select_one("span[property='v:itemreviewed']").text.strip() if soup.select_one("span[property='v:itemreviewed']") else None, #extract_list("导演"),
        "writers": extract_list("编剧"),
        "cast": extract_list("主演"),
        "genre": extract_text("类型:"),
        "country": extract_text("制片国家/地区:"),
        "language": extract_text("语言:"),
        "release_dates": [span.get_text(strip=True) for span in info_div.find_all("span", property="v:initialReleaseDate")],
        "duration": extract_text("片长:"),
        "aka": extract_text("又名:"),
        "imdb": extract_text("IMDb:"),
        "rating": rating,
        "review_count": review_count
    }

    return metadata

In [9]:
def get_douban_metadata(title):
    soup = search_douban_mobile(title)
    results = extract_top_results(soup)
    matched_url = find_exact_match(results, title)
    print("Matched URL:", matched_url)
    if matched_url:
        return parse_douban_info(matched_url)
    else:
        print("Exact match not found.")
        return None

In [12]:
title = "霸王别姬(京剧)"
metadata = get_douban_metadata(title)
print(metadata)

Matched URL: https://m.douban.com/movie/subject/20645019/
{'title': '霸王别姬(京剧)', 'matched_url': 'https://m.douban.com/movie/subject/20645019/', 'director': '霸王别姬(京剧)', 'writers': ['黎中城', '王涌石'], 'cast': ['尚长荣', '史依弘', '杨东虎', '蓝天', '李军', '王立军', '金喜全', '傅希如', '任广平', '高明博', '徐建忠', '陈宇'], 'genre': '', 'country': '中国大陆', 'language': '汉语普通话', 'release_dates': ['2015-12-05(中国大陆)', '2014-05-30(美国)'], 'duration': '', 'aka': '霸王别姬(戏曲电影) / 霸王别姬3D / 霸王别姬 / Farewell My Concubine: the Beijing Opera', 'imdb': 'tt7435474', 'rating': '9.0', 'review_count': '2788'}


In [15]:
title = "赴山海"
metadata = get_douban_metadata(title)
print(metadata)

Matched URL: https://m.douban.com/group/744622/
{}


In [ ]:
import time

time.sleep(5)
